# 💾 Module 06: Model Serialization (.pkl) & Interactive Inference
### Production PKL Packaging and Single-Scan Diagnostic Evaluation

This notebook implements:
1. Exporting trained PyTorch models, state dictionaries, evaluation metrics, and clinical metadata into `.pkl` format.
2. Deserializing and validating the `.pkl` package.
3. Interactive end-to-end single-image diagnostic function:
   - OCT Scan Preprocessing
   - Disease Category Prediction & Full Softmax Distribution
   - Explainable Grad-CAM Heatmap Overlay
   - Clinical Pathology Summary Report

In [ ]:
import os
import io
import pickle
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn.functional as F

from src.dataset import CLASS_NAMES, CLASS_DESCRIPTIONS, get_transforms
from src.gradcam import GradCAM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute Device: {device}")

### 6.1 Model Packaging to .pkl Format

In [ ]:
def save_model_package_pkl(model, model_name, metrics, filepath, extra_info=None):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    # Remove registered hooks before pickling
    for m in model.modules():
        if hasattr(m, "_forward_hooks"): m._forward_hooks.clear()
        if hasattr(m, "_backward_hooks"): m._backward_hooks.clear()
        if hasattr(m, "_forward_pre_hooks"): m._forward_pre_hooks.clear()
        if hasattr(m, "_full_backward_hooks"): m._full_backward_hooks.clear()

    package = {
        "model_name": model_name,
        "model_architecture": str(model.__class__.__name__),
        "state_dict": model.state_dict(),
        "model": model,
        "metrics": metrics,
        "class_names": CLASS_NAMES,
        "input_size": (3, 224, 224),
        "extra_info": extra_info or {}
    }
    with open(filepath, "wb") as f:
        pickle.dump(package, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Successfully serialized model package to PKL: {filepath}")

# Load and verify existing PKL package
pkl_path = "models/best_retinal_oct_model.pkl"
with open(pkl_path, "rb") as f:
    pkg = pickle.load(f)

print("Loaded Model Package Metadata:")
print(f"  Model Architecture : {pkg['model_architecture']}")
print(f"  Model Name         : {pkg['model_name']}")
print(f"  Accuracy           : {pkg['metrics']['Accuracy']*100:.2f}%")
print(f"  Macro F1-Score     : {pkg['metrics']['F1-Score']:.4f}")
print(f"  Classes            : {pkg['class_names']}")

### 6.2 Interactive Diagnostic Inference Function

In [ ]:
def diagnose_oct_scan(image_input, model=None):
    """Performs end-to-end diagnostic inference and Grad-CAM explainability on an OCT image."""
    if model is None:
        with open("models/best_retinal_oct_model.pkl", "rb") as f:
            model = pickle.load(f)["model"]
    model = model.to(device)

    if isinstance(image_input, str):
        pil_img = Image.open(image_input).convert("RGB")
    elif isinstance(image_input, bytes):
        pil_img = Image.open(io.BytesIO(image_input)).convert("RGB")
    else:
        pil_img = image_input.convert("RGB")

    _, eval_tf = get_transforms()
    tensor_img = eval_tf(pil_img).unsqueeze(0).to(device)

    gradcam = GradCAM(model)
    heatmap, pred_c, conf, probs = gradcam.generate_heatmap(tensor_img)
    blended, _ = GradCAM.overlay_heatmap_on_image(pil_img, heatmap, alpha=0.45)
    gradcam.remove_hooks()

    pred_class_name = CLASS_NAMES[pred_c]

    # Visualization
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4.5))

    ax1.imshow(pil_img, cmap="gray")
    ax1.set_title("Input OCT B-Scan", fontsize=11, fontweight="bold")
    ax1.axis("off")

    ax2.imshow(blended)
    ax2.set_title(f"Grad-CAM: {pred_class_name} ({conf*100:.1f}%)", fontsize=11, fontweight="bold")
    ax2.axis("off")

    colors = ["#d9534f" if i == pred_c else "#428bca" for i in range(len(CLASS_NAMES))]
    y_pos = np.arange(len(CLASS_NAMES))
    ax3.barh(y_pos, probs * 100, color=colors)
    ax3.set_yticks(y_pos)
    ax3.set_yticklabels(CLASS_NAMES, fontsize=10, fontweight="bold")
    ax3.set_xlabel("Predicted Probability (%)", fontsize=10)
    ax3.set_title("Class Probability Distribution", fontsize=11, fontweight="bold")
    ax3.set_xlim(0, 100)
    for i, v in enumerate(probs * 100):
        ax3.text(v + 1.5, i, f"{v:.1f}%", va="center", fontsize=9, fontweight="bold")

    plt.tight_layout()
    plt.show()

    print(f"Diagnostic Result    : {pred_class_name} ({conf*100:.2f}% Confidence)")
    print(f"Pathology Description: {CLASS_DESCRIPTIONS[pred_class_name]}")

### 6.3 Diagnostic Tests on Clinical Sample Scans

In [ ]:
sample_scans = [
    "assets/sample_images/CNV_sample_1.jpeg",
    "assets/sample_images/DME_sample_1.jpeg",
    "assets/sample_images/DRUSEN_sample_1.jpeg",
    "assets/sample_images/NORMAL_sample_1.jpeg"
]

for scan_path in sample_scans:
    if os.path.exists(scan_path):
        print("=" * 70)
        print(f"Testing on: {scan_path}")
        diagnose_oct_scan(scan_path)
        print("=" * 70)